In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# C2 2A: second-axis three-arm diagnostic

This is a new fixed-terminal paired localization measurement, not recovery of the earlier six-arm terminal tensor. It retains only `ZERO`, `PLUS_E2`, and `MINUS_E2`; all share one newly generated ordinary Wan terminal. It persists the terminal, actual post-FP32-write blocks, lossless float-RGB tensors, both q layers, MP4 files, configuration, and retained failures in a new Drive directory.

In [ ]:
from pathlib import Path
import sys, subprocess
REPOSITORY_URL = 'https://github.com/RICHAAARC/SC-SSTW.git'
SOURCE_BRANCH = 'c2a-2a-colab-preparation'
SOURCE = Path('/content/c2a_second_axis_diagnostic_source')
if SOURCE.exists():
    raise FileExistsError('Use a fresh runtime; preserve existing source')
subprocess.run(['git', 'init', str(SOURCE)], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'remote', 'add', 'origin', REPOSITORY_URL], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'fetch', '--depth', '1', 'origin', SOURCE_BRANCH], check=True)
subprocess.run(['git', '-C', str(SOURCE), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
print('Source:', subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip())


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers', 'transformers', 'accelerate', 'ftfy', 'sentencepiece', 'safetensors', 'huggingface_hub', 'numpy', 'Pillow'], check=True)
subprocess.run(['ffmpeg', '-version'], check=True)
# Keep Colab CUDA PyTorch; actual versions are recorded by the result.


## Fixed diagnostic configuration

The reused ordinary terminal path remains Wan2.1 T2V 1.3B, the cube prompt, seed 1275, 320x512, 49 frames, 50 steps and CFG 5. The write/read carrier remains beta 0.25, rho 0.5, channels 0/1, central 8x8 support, groups 1/2/3 written and group 2 read. Expected real-work calls are one terminal generation with 100 transformer forwards, three VAE decodes and six VAE encodes.

In [ ]:
from datetime import datetime, timezone
CONFIG = SOURCE / 'runtime/c2a/c2a_second_axis_diagnostic_run.json'
RUN_ID = 'c2a_second_axis_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/C2A_SecondAxis_Diagnostic') / RUN_ID
print(CONFIG.read_text())
print('Output:', OUTPUT)
if OUTPUT.exists():
    raise FileExistsError(str(OUTPUT))


In [ ]:
import os, signal
command = [sys.executable, '-m', 'runtime.c2a.run_second_axis_diagnostic', '--config', str(CONFIG), '--output', str(OUTPUT), '--execute']
process = subprocess.Popen(command, cwd=SOURCE, start_new_session=True)
try:
    returncode = process.wait()
except BaseException:
    try: process.send_signal(signal.SIGTERM)
    except ProcessLookupError: pass
    try: process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        try: os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError: pass
        process.wait()
    raise
print('launcher exit', returncode)
print((OUTPUT / 'result.json').read_text() if (OUTPUT / 'result.json').exists() else 'No result file')
if returncode: raise subprocess.CalledProcessError(returncode, command)


## Persisted diagnostic package

The enabled run writes only to `MyDrive/Video-WM/C2A_SecondAxis_Diagnostic/<UTC-run-id>/`. It never overwrites the six-arm result. `shared_terminal_normalized.pt` is CPU/original-dtype; `actual_written_blocks/<arm>.pt` contains the actual decoded FP32 support blocks for groups 1/2/3; `pre_codec_float_rgb/<arm>.pt` is an uncompressed float tensor. `result.json` records actual post-write covariances/q, pre-codec q, post-MP4 q, O2/M2 vectors/norms, counts, and retained failures. No O2/M2 ratio is reported when the odd response is small.